# NHL-Beyond-27 · Book 3 — Corsi Composites & Spicy (z) Analysis

Repo: `ewnike/NHL-Beyond-27` — *MADS Milestone I Project*
Python: 3.13.7 (pyenv env: `nhl_beyond27-3.13.7`)
Editors/Tools: VSCode, Git/GitHub, Postgres + pgAdmin
Logs: `log_utils.py` → `logs/`

**This notebook covers:**

1. Load analysis view/table
2. Define roles (Defense vs Forwards)
3. **Part A — Composite Corsi on z-scores**: unweighted vs role-weighted (D/F) → visuals → regression
4. **Part B — Spicy (within-player z)**: definition → visuals → regression
5. **Part C — Spicy-Weighted (within-player z)**: definition → visuals → regression
6. (Optional) Export small summary CSVs for the paper

**Rel-age order used throughout:** −2, −1, 0, 1, 2 (peak = 0).
We keep **composite Corsi on z-scores** (Part A) separate from **Spicy** metrics (Parts B/C).



**Load necessary imports**

In [1]:
import sys

import numpy as np
import pandas as pd

# Optional: these are needed later; harmless to import now
import plotly.express
import plotly.graph_objects as go
import statsmodels.api as sm
import statsmodels.formula.api as smf

print(
    "py",
    sys.version.split()[0],
    "| numpy",
    np.__version__,
    "| pandas",
    pd.__version__,
    "| plotly",
    plotly.__version__,
    "| statsmodels",
    sm.__version__,
)

py 3.13.7 | numpy 2.3.3 | pandas 2.3.2 | plotly 6.3.0 | statsmodels 0.14.5


**Load peak player season table csv and load z-table and load z-cohort-table from DB**

In [2]:
import contextlib
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import statsmodels.api as sm  # ok to keep if used later


# ---------- loaders ----------
def load_z_table():
    try:
        from db_utils import get_db_engine
        eng = get_db_engine()
        df = pd.read_sql("SELECT * FROM public.player_five_year_aligned_z", eng)
        print("Loaded z-table from DB:", len(df))
        return df
    except Exception as e:
        print("[Info] DB not available or table missing; using CSV fallback:", e)
        csv = Path("data/outputs/player_five_year_aligned_z.csv")
        assert csv.exists(), "CSV fallback not found: data/outputs/player_five_year_aligned_z.csv"
        df = pd.read_csv(csv)
        print("Loaded z-table from CSV:", len(df))
        return df

def load_peak_table():
    try:
        from db_utils import get_db_engine
        eng = get_db_engine()
        q = (
            'SELECT player, season, position, "CF%", "CF/60", "CA/60" '
            "FROM public.player_peak_season"
        )
        df = pd.read_sql(q, eng)
        print("Loaded peak table from DB:", len(df))
        return df
    except Exception as e:
        print("[Info] DB not available; using CSV fallback:", e)
        csv = Path("data/peak_player_season_stats.csv")
        assert csv.exists(), "CSV fallback not found: data/peak_player_season_stats.csv"
        df = pd.read_csv(csv)
        print("Loaded peak table from CSV:", len(df))
        return df

def load_cohort_z_table():
    try:
        from db_utils import get_db_engine
        eng = get_db_engine()
        df = pd.read_sql("SELECT * FROM public.player_five_year_aligned_z_cohort", eng)
        print("Loaded z-cohort-table from DB:", len(df))
        return df
    except Exception as e:
        print("[Info] DB not available or table missing; using CSV fallback:", e)
        csv = Path("data/outputs/player_five_year_aligned_z_cohort.csv")
        assert csv.exists(), "CSV fallback not found: data/outputs/player_five_year_aligned_z_cohort.csv"
        df = pd.read_csv(csv)
        print("Loaded z-cohort-table from CSV:", len(df))
        return df

# ---------- normalize columns ----------
def _norm_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    import re
    def norm(c):
        c = str(c).strip().lower()
        c = c.replace("%", "pct").replace("/", "_per_")
        return re.sub(r"[^a-z0-9]+", "_", c).strip("_")
    df.columns = [norm(c) for c in df.columns]
    return df

# ---------- role normalization ----------
def to_role_fd(x: str) -> str:
    s = str(x).strip().upper()
    return "D" if s.startswith("D") else "F"

def normalize_roles_inplace(df: pd.DataFrame, role_col_candidates=("role","position","pos")) -> pd.DataFrame:
    """
    Ensures df has a 'role' column with only 'F' or 'D'.
    """
    for c in role_col_candidates:
        if c in df.columns:
            df["role"] = df[c].map(to_role_fd)
            return df
    raise KeyError("Need a role-like column ('role', 'position', or 'pos') to derive F/D.")

# ---------- load ----------
df_z        = _norm_cols(load_z_table())
df_raw      = _norm_cols(load_peak_table())
df_z_cohort = _norm_cols(load_cohort_z_table())

# If the CSV uses 'pos', rename BEFORE role normalization so checks pass
alias_map = {"pos": "position"}
df_z.rename(columns={k: v for k, v in alias_map.items() if k in df_z.columns}, inplace=True)
df_raw.rename(columns={k: v for k, v in alias_map.items() if k in df_raw.columns}, inplace=True)
df_z_cohort.rename(columns={k: v for k, v in alias_map.items() if k in df_z_cohort.columns}, inplace=True)

# Normalize roles (creates/overwrites 'role' as F/D)
normalize_roles_inplace(df_z)
normalize_roles_inplace(df_z_cohort)
# (optional) if you'll color df_raw by role later:
with contextlib.suppress(KeyError):
    normalize_roles_inplace(df_raw)

# rel_age categories (if present)
ORDER = [-2, -1, 0, 1, 2]
for df in (df_z, df_z_cohort):
    if "rel_age" in df.columns:
        df["rel_age"] = pd.Categorical(
            pd.to_numeric(df["rel_age"], errors="coerce"),
            categories=ORDER, ordered=True
        )

print("Ready. (z cols):", list(df_z.columns)[:12], "…")
print("Ready. (raw cols):", list(df_raw.columns)[:12], "…")
print("Ready. (z cohort cols):", list(df_z_cohort.columns)[:12], "…")


2025-10-02 00:48:21,779 - INFO - db_utils - Using DATABASE_URL from environment.
2025-10-02 00:48:21,841 - INFO - db_utils - Using DATABASE_URL from environment.
2025-10-02 00:48:21,860 - INFO - db_utils - Using DATABASE_URL from environment.


Loaded z-table from DB: 1410
[Info] DB not available; using CSV fallback: sqlalchemy.cyextension.immutabledict.immutabledict is not a sequence
Loaded peak table from CSV: 3121
Loaded z-cohort-table from DB: 1410
Ready. (z cols): ['player', 'position', 'peak_year', 'rel_age', 'start_year', 'season', 'age', 'cf_pct', 'cf60', 'ca60', 'cf_pct_z', 'cf60_z'] …
Ready. (raw cols): ['player', 'eh_id', 'api_id', 'season', 'team', 'position', 'shoots', 'birthday', 'age', 'draft_yr', 'draft_rd', 'draft_ov'] …
Ready. (z cohort cols): ['player', 'position', 'peak_year', 'rel_age', 'start_year', 'season', 'age', 'cf_pct', 'cf60', 'ca60', 'cf_pct_z', 'cf60_z'] …


## Part A — Composite Corsi on Z-scores (Unweighted vs Role-Weighted)



**Goal.** Summarize play-driving with a single standardized composite, built from **within-player z-scores** (each metric centered/scaled by that player’s 5-year baseline):

* `cf_pct_z` — possession share (CF%)
* `cf60_z` — shot creation per 60 (CF/60)
* `ca60_z` — shot suppression per 60 (CA/60) *(enters with a minus sign)*

### Two composites

**1) Unweighted composite z**

z_comp_unweighted = mean(cf_pct_z, cf60_z, -ca60_z)


**2) Role-weighted composite z** *(simple, fixed heuristics for this milestone)*

* **Defense (D):** emphasize suppression a bit more; creation a bit less

z_comp_weighted = 0.5*cf_pct_z + 0.2*cf60_z - 0.3*ca60_z


* **Forwards (F):** emphasize creation a bit more; suppression a bit less

z_comp_weighted = 0.5*cf_pct_z + 0.3*cf60_z - 0.2*ca60_z


> **Why z-scores?** They avoid raw-scale mixing and make the composite interpretable as “high/low **relative to the same player’s baseline**.”
> **Why these weights?** Clear, documented heuristics to keep Book 3 simple. You can tune them later with data-driven optimization or cross-validation.

### What to look for in the plots

* **By `rel_age` (−2, −1, 0, +1, +2):** trajectories around peak (0) for Defense vs Forwards.
* **Unweighted vs role-weighted:** how weighting shifts the relative separation of roles.
* **CI ribbons:** 95% confidence intervals around the mean composite by role × `rel_age`.

### Outputs in this part

* Line charts of **mean composite z** by role across `rel_age` with **95% CI** ribbons.
* Histograms / boxplots comparing the distribution of unweighted vs role-weighted composites by role.
* (Optional) Export of summary tables used to render the figures.



In [3]:
import numpy as np
import pandas as pd

dfz = df_z.copy()

# Role from position
dfz["role"] = np.where(dfz["position"].astype(str).str.upper().str.startswith("D"), "D", "F")


# Helper: mean of available values
def _mean_available(vals):
    v = [x for x in vals if pd.notna(x)]
    return float(np.mean(v)) if v else np.nan


# Unweighted composite on z-scores
def comp_unw(r):
    return _mean_available([r["cf_pct_z"], r["cf60_z"], -r["ca60_z"]])


# Role-weighted composite on z-scores
W_D = {"cf_pct_z": 0.5, "cf60_z": 0.2, "ca60_z": 0.3}
W_F = {"cf_pct_z": 0.5, "cf60_z": 0.3, "ca60_z": 0.2}


def comp_w(r):
    w = W_D if r["role"] == "D" else W_F
    num, den = 0.0, 0.0
    for k, wt in w.items():
        v = r.get(k)
        if pd.notna(v):
            if k == "ca60_z":
                v = -v  # suppress against
            num += wt * v
            den += wt
    return num / den if den else np.nan


dfz["z_comp_unweighted"] = dfz.apply(comp_unw, axis=1)
dfz["z_comp_weighted"] = dfz.apply(comp_w, axis=1)

# Keep rel_age ordered for plots
order = [-2, -1, 0, 1, 2]
if "rel_age" in dfz.columns:
    dfz["rel_age"] = pd.Categorical(
        pd.to_numeric(dfz["rel_age"], errors="coerce"), categories=order, ordered=True
    )

dfz[["player", "season", "role", "rel_age", "z_comp_unweighted", "z_comp_weighted"]].head(8)

,player,season,role,rel_age,z_comp_unweighted,z_comp_weighted
0,Adam Henrique,16-17,F,-1,-0.405994,-0.428285
1,Adam Henrique,19-20,F,2,0.778992,0.931875
2,Adam Henrique,15-16,F,-2,-0.232633,-0.538525
3,Adam Henrique,17-18,F,0,0.520218,0.683384
4,Adam Henrique,18-19,F,1,-0.660584,-0.648449
5,Adam Larsson,20-21,D,0,-0.557387,-0.552449
6,Adam Larsson,22-23,D,2,1.471390,1.507333
7,Adam Larsson,21-22,D,1,0.027213,0.044183


In [4]:
import numpy as np
import pandas as pd
from plotly import colors as pcolors


def hex_to_rgba(hex_color: str, alpha: float = 0.2) -> str:
    """Convert '#RRGGBB' to 'rgba(r,g,b,alpha)'."""
    h = hex_color.lstrip("#")
    if len(h) != 6:
        raise ValueError(f"Expected 6-digit hex like '#RRGGBB', got {hex_color}")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def mean_ci(df: pd.DataFrame, col: str) -> pd.DataFrame:
    if not {"role", "rel_age", col}.issubset(df.columns):
        missing = {"role", "rel_age", col} - set(df.columns)
        raise KeyError(f"Column(s) missing from df: {missing}")

    g = (
        df.groupby(["role", "rel_age"])
        .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
        .reset_index()
    )
    # handle n==1 (sd is NaN) safely
    g["se"] = g["sd"].fillna(0) / np.sqrt(g["n"].clip(lower=1))
    g["lower"] = g["mean"] - 1.96 * g["se"]
    g["upper"] = g["mean"] + 1.96 * g["se"]
    return g


def ribbon_line(means: pd.DataFrame, title: str, ylab: str) -> go.Figure:
    fig = go.Figure()

    # Default palette with fallbacks for unseen roles
    base_palette = {"D": "#1f77b4", "F": "#ff7f0e"}
    fallback_cycle = pcolors.qualitative.Plotly  # a list of hex colors
    cycle_idx = 0

    # ensure rel_age is numeric for sorting; keep original for axis categories
    rel_age_order = [-2, -1, 0, 1, 2]
    means = means.copy()
    means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")

    # map roles to colors with stable fallback
    role_colors = {}
    for role in means["role"].dropna().unique():
        if role in base_palette:
            role_colors[role] = base_palette[role]
        else:
            role_colors[role] = fallback_cycle[cycle_idx % len(fallback_cycle)]
            cycle_idx += 1

    for role in means["role"].dropna().unique():
        sub = means.loc[means["role"] == role].sort_values("rel_age")
        sub = sub[sub["rel_age"].isin(rel_age_order)]  # keep known x
        if sub.empty:
            continue

        # use strings for categorical x so categoryarray works
        x = sub["rel_age"].astype(int).astype(str)
        c = role_colors[role]

        # Upper bound (invisible line to anchor fill)
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["upper"],
                line=dict(width=0),
                hoverinfo="skip",
                showlegend=False,
                name=f"{role} 95% CI (upper)",
            )
        )

        # Lower bound + fill between lower and the previous (upper) trace
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["lower"],
                line=dict(width=0),
                hoverinfo="skip",
                fill="tonexty",
                fillcolor=hex_to_rgba(c, alpha=0.2),
                showlegend=False,
                name=f"{role} 95% CI",
            )
        )

        # Mean line
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=c, width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    # y-range (guard against all-NaN)
    lower_min = np.nanmin(means["lower"].to_numpy()) if "lower" in means else np.nan
    upper_max = np.nanmax(means["upper"].to_numpy()) if "upper" in means else np.nan
    if np.isfinite(lower_min) and np.isfinite(upper_max):
        y_min = float(np.floor(lower_min - 0.2))
        y_max = float(np.ceil(upper_max + 0.2))
        y_range = [y_min, y_max]
    else:
        y_range = None  # let Plotly autoscale

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(categoryorder="array", categoryarray=["-2", "-1", "0", "1", "2"])
    if y_range:
        fig.update_yaxes(range=y_range, dtick=0.2)
    else:
        fig.update_yaxes(dtick=0.2)

    return fig


In [5]:
# Unweighted
fig_h1 = px.histogram(
    dfz,
    x="z_comp_unweighted",
    color="role",
    barmode="overlay",
    nbins=40,
    opacity=0.6,
    title="Composite z (Unweighted) — Distribution by Role",
    labels={"z_comp_unweighted": "unweighted composite z"},
)
fig_h1.show()

# Weighted
fig_h2 = px.histogram(
    dfz,
    x="z_comp_weighted",
    color="role",
    barmode="overlay",
    nbins=40,
    opacity=0.6,
    title="Composite z (Role-weighted) — Distribution by Role",
    labels={"z_comp_weighted": "role-weighted composite z"},
)
fig_h2.show()

In [6]:
# Unweighted
fig_b1 = px.box(
    dfz,
    x="role",
    y="z_comp_unweighted",
    title="Composite z (Unweighted) — Boxplot by Role",
    labels={"z_comp_unweighted": "unweighted composite z"},
)
fig_b1.show()

# Weighted
fig_b2 = px.box(
    dfz,
    x="role",
    y="z_comp_weighted",
    title="Composite z (Role-weighted) — Boxplot by Role",
    labels={"z_comp_weighted": "role-weighted composite z"},
)
fig_b2.show()

In [7]:
import numpy as np
import plotly.graph_objects as go


def hex_to_rgba(hex_color: str, alpha: float = 0.2) -> str:
    """Convert '#RRGGBB' to 'rgba(r,g,b,alpha)'."""
    h = hex_color.lstrip("#")
    r = int(h[0:2], 16)
    g = int(h[2:4], 16)
    b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"


def mean_ci(df, col):
    g = (
        df.groupby(["role", "rel_age"], observed=True)
        .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
        .reset_index()
    )
    g["se"] = g["sd"] / np.sqrt(g["n"].clip(lower=1))
    g["lower"] = g["mean"] - 1.96 * g["se"]
    g["upper"] = g["mean"] + 1.96 * g["se"]
    return g


def ribbon_line(means, title, ylab, ytick=0.05, pad=0.05):
    fig = go.Figure()
    palette = {"D": "#1f77b4", "F": "#ff7f0e"}

    means = means.copy()
    if "rel_age" in means.columns:
        means["rel_age"] = pd.to_numeric(means["rel_age"], errors="coerce")

    for role in means["role"].dropna().unique():
        sub = means[means["role"] == role].sort_values("rel_age")
        if sub.empty:
            continue
        x = sub["rel_age"].astype(int).astype(str)
        fig.add_trace(
            go.Scatter(
                x=x,
                y=sub["mean"],
                mode="lines+markers",
                line=dict(color=palette.get(role, "#888888"), width=2),
                marker=dict(size=7),
                name=f"{role} mean",
            )
        )

    # Compute a tight range around the data (including CI if present)
    lo = np.nanmin(np.column_stack([means.get("lower", means["mean"]).to_numpy()]))
    hi = np.nanmax(np.column_stack([means.get("upper", means["mean"]).to_numpy()]))
    span = float(hi - lo) if np.isfinite(hi - lo) else 0.0
    pad_abs = max(span * pad, 0.02)  # ensure a tiny buffer
    y_range = [float(lo - pad_abs), float(hi + pad_abs)] if np.isfinite(span) else None

    fig.update_layout(
        title=title,
        xaxis_title="rel_age",
        yaxis_title=ylab,
        template="plotly_white",
        legend=dict(title=""),
        height=420,
    )
    fig.update_xaxes(categoryorder="array", categoryarray=["-2", "-1", "0", "1", "2"])
    fig.update_yaxes(
        range=y_range,  # tight zoom
        dtick=ytick,  # finer grid: try 0.05 or 0.02
        tickformat=".2f",  # more precise labels
        zeroline=True,  # accent reference
        zerolinewidth=1,
        gridwidth=0.5,
    )
    return fig


# Build tables and plot
m_unw = mean_ci(dfz, "z_comp_unweighted")
m_w = mean_ci(dfz, "z_comp_weighted")

ribbon_line(m_unw, "Composite z (Unweighted) — Mean by rel_age (95% CI)", "unweighted z").show()
ribbon_line(m_w, "Composite z (Role-weighted) — Mean by rel_age (95% CI)", "role-weighted z").show()

regression analysis for corsi composites

In [8]:
# --- OLS: composite z (unweighted & role-weighted) ~ role * rel_age (categorical), HC3 robust SEs

import numpy as np
import pandas as pd

# Work on a copy
zz = dfz.copy()

# Ensure role exists (D vs F) if not already present
if "role" not in zz.columns:
    zz["role"] = np.where(zz["position"].astype(str).str.upper().str.startswith("D"), "D", "F")

# Enforce rel_age categorical ordering if present
order = [-2, -1, 0, 1, 2]
if "rel_age" in zz.columns:
    zz["rel_age"] = pd.Categorical(
        pd.to_numeric(zz["rel_age"], errors="coerce"),
        categories=order,
        ordered=True,
    )


def run_ols(ycol: str, data: pd.DataFrame):
    """
    Fit OLS with HC3 robust SEs.
    If rel_age exists, use interaction: y ~ C(role) * C(rel_age, Treatment(0))
    Otherwise: y ~ C(role)
    Returns (model, tidy_coefs_df).
    """
    if "rel_age" in data.columns and data["rel_age"].notna().any():
        formula = f"{ycol} ~ C(role) * C(rel_age, Treatment(0))"
    else:
        formula = f"{ycol} ~ C(role)"

    model = smf.ols(formula, data=data).fit(cov_type="HC3")
    print(f"\n=== OLS (HC3) for {ycol} ===")
    print(model.summary())

    # Tidy coef table (coef, SE, p, 95% CI) + R^2 for convenience
    coefs = pd.DataFrame(
        {
            "term": model.params.index,
            "coef": model.params.values,
            "se": model.bse.values,
            "p": model.pvalues.values,
        }
    )
    ci = model.conf_int()
    coefs["ci_lo"] = ci[0].values
    coefs["ci_hi"] = ci[1].values
    coefs["r2"] = model.rsquared
    coefs["r2_adj"] = model.rsquared_adj
    return model, coefs


# Run for both composites
m_unw, coefs_unw = run_ols("z_comp_unweighted", zz)
m_w, coefs_w = run_ols("z_comp_weighted", zz)

# Show tidy tables
display(coefs_unw)
display(coefs_w)

# Optional: marginal predictions by role × rel_age (95% CI) for plotting
if "rel_age" in zz.columns:
    grid = pd.MultiIndex.from_product([["D", "F"], order], names=["role", "rel_age"]).to_frame(
        index=False
    )
    pred_unw = m_unw.get_prediction(grid).summary_frame(alpha=0.05)
    pred_w = m_w.get_prediction(grid).summary_frame(alpha=0.05)

    pred_unw = pd.concat([grid, pred_unw[["mean", "mean_ci_lower", "mean_ci_upper"]]], axis=1)
    pred_w = pd.concat([grid, pred_w[["mean", "mean_ci_lower", "mean_ci_upper"]]], axis=1)

    pred_unw.rename(
        columns={"mean": "y", "mean_ci_lower": "ci_lo", "mean_ci_upper": "ci_hi"}, inplace=True
    )
    pred_w.rename(
        columns={"mean": "y", "mean_ci_lower": "ci_lo", "mean_ci_upper": "ci_hi"}, inplace=True
    )

    print("\nPredictions (unweighted):")
    display(pred_unw)

    print("\nPredictions (role-weighted):")
    display(pred_w)


=== OLS (HC3) for z_comp_unweighted ===
                            OLS Regression Results                            
Dep. Variable:      z_comp_unweighted   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     1.816
Date:                Thu, 02 Oct 2025   Prob (F-statistic):             0.0610
Time:                        00:48:22   Log-Likelihood:                -1588.3
No. Observations:                1410   AIC:                             3197.
Df Residuals:                    1400   BIC:                             3249.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------

,term,coef,se,p,ci_lo,ci_hi,r2,r2_adj
0,Intercept,-0.009147,0.067207,0.891746,-0.140870,0.122577,0.012098,0.005747
1,C(role)[T.F],0.009825,0.087905,0.911010,-0.162466,0.182115,0.012098,0.005747
2,"C(rel_age, Treatment(0))[T.-2]",0.075788,0.108040,0.483002,-0.135966,0.287542,0.012098,0.005747
3,"C(rel_age, Treatment(0))[T.-1]",0.069372,0.096981,0.474413,-0.120707,0.259452,0.012098,0.005747
4,"C(rel_age, Treatment(0))[T.1]",-0.008445,0.103823,0.935173,-0.211934,0.195044,0.012098,0.005747
5,"C(rel_age, Treatment(0))[T.2]",-0.090983,0.100895,0.367183,-0.288733,0.106767,0.012098,0.005747
6,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-2]",-0.068091,0.135083,0.614213,-0.332848,0.196666,0.012098,0.005747
7,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-1]",0.053009,0.123868,0.668686,-0.189767,0.295786,0.012098,0.005747
8,"C(role)[T.F]:C(rel_age, Treatment(0))[T.1]",0.039235,0.128253,0.759668,-0.212137,0.290606,0.012098,0.005747
9,"C(role)[T.F]:C(rel_age, Treatment(0))[T.2]",-0.073276,0.130671,0.574955,-0.329387,0.182834,0.012098,0.005747


,term,coef,se,p,ci_lo,ci_hi,r2,r2_adj
0,Intercept,-0.013688,0.070520,0.846097,-0.151905,0.124529,0.012146,0.005795
1,C(role)[T.F],0.004032,0.091895,0.965007,-0.176079,0.184142,0.012146,0.005795
2,"C(rel_age, Treatment(0))[T.-2]",0.088182,0.112975,0.435070,-0.133245,0.309609,0.012146,0.005795
3,"C(rel_age, Treatment(0))[T.-1]",0.088050,0.100957,0.383129,-0.109823,0.285923,0.012146,0.005795
4,"C(rel_age, Treatment(0))[T.1]",-0.013579,0.108420,0.900331,-0.226078,0.198920,0.012146,0.005795
5,"C(rel_age, Treatment(0))[T.2]",-0.094213,0.105886,0.373599,-0.301746,0.113321,0.012146,0.005795
6,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-2]",-0.072322,0.141202,0.608519,-0.349074,0.204429,0.012146,0.005795
7,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-1]",0.050575,0.129088,0.695216,-0.202432,0.303582,0.012146,0.005795
8,"C(role)[T.F]:C(rel_age, Treatment(0))[T.1]",0.062279,0.134117,0.642388,-0.200585,0.325143,0.012146,0.005795
9,"C(role)[T.F]:C(rel_age, Treatment(0))[T.2]",-0.060689,0.136862,0.657452,-0.328934,0.207555,0.012146,0.005795



Predictions (unweighted):


,role,rel_age,y,ci_lo,ci_hi
0,D,-2,0.066641,-0.099156,0.232439
1,D,-1,0.060226,-0.076811,0.197263
2,D,0,-0.009147,-0.140870,0.122577
3,D,1,-0.017591,-0.172693,0.137511
4,D,2,-0.100130,-0.247622,0.047363
5,F,-2,0.008375,-0.105307,0.122057
6,F,-1,0.123060,0.020702,0.225418
7,F,0,0.000678,-0.110376,0.111732
8,F,1,0.031468,-0.065727,0.128663
9,F,2,-0.163581,-0.282554,-0.044608



Predictions (role-weighted):


,role,rel_age,y,ci_lo,ci_hi
0,D,-2,0.074494,-0.098497,0.247485
1,D,-1,0.074362,-0.067236,0.215959
2,D,0,-0.013688,-0.151905,0.124529
3,D,1,-0.027267,-0.188673,0.134139
4,D,2,-0.107901,-0.262710,0.046909
5,F,-2,0.006203,-0.113068,0.125475
6,F,-1,0.128968,0.021623,0.236313
7,F,0,-0.009656,-0.125137,0.105825
8,F,1,0.039043,-0.063940,0.142027
9,F,2,-0.164558,-0.289255,-0.039862


** Quick note on negatives **
Your composites are z-scores (within-player). It’s normal for means to sit near 0 and be negative/positive depending on the season relative to the player’s baseline. That’s expected behavior, not an error.

**Export table information to data/outputs

Part A2 The same but for player vs cohort

In [9]:
def add_composites(df: pd.DataFrame) -> pd.DataFrame:
    """Ensure role (F/D) and compute z_comp_unweighted & z_comp_weighted from z-cols."""
    out = df.copy()

    # role (F/D)
    if "role" not in out.columns:
        if "position" in out.columns:
            out["role"] = np.where(out["position"].astype(str).str.upper().str.startswith("D"), "D", "F")
        else:
            raise KeyError("Need 'role' or 'position' to derive F/D roles")

    # helper: mean of available parts
    def _mean_available(vals):
        v = [x for x in vals if pd.notna(x)]
        return float(np.mean(v)) if v else np.nan

    # unweighted composite on z-scores
    def comp_unw(r):
        return _mean_available([r.get("cf_pct_z"), r.get("cf60_z"), -r.get("ca60_z")])

    # role-weighted composite on z-scores
    W_D = {"cf_pct_z": 0.5, "cf60_z": 0.2, "ca60_z": 0.3}
    W_F = {"cf_pct_z": 0.5, "cf60_z": 0.3, "ca60_z": 0.2}

    def comp_w(r):
        w = W_D if r["role"] == "D" else W_F
        num = den = 0.0
        for k, wt in w.items():
            v = r.get(k)
            if pd.notna(v):
                if k == "ca60_z":  # suppression enters with negative sign
                    v = -v
                num += wt * v
                den += wt
        return num / den if den else np.nan

    out["z_comp_unweighted"] = out.apply(comp_unw, axis=1)
    out["z_comp_weighted"]  = out.apply(comp_w, axis=1)

    # rel_age ordering
    if "rel_age" in out.columns:
        order = [-2, -1, 0, 1, 2]
        out["rel_age"] = pd.Categorical(pd.to_numeric(out["rel_age"], errors="coerce"),
                                        categories=order, ordered=True)
    return out


In [10]:
# ===== Build SELF & COHORT composite frames =====
# SELF uses df_z (already contains within-player z’s)
dfz_self = add_composites(df_z)

# COHORT uses df_z_cohort: ensure cohort z’s exist or compute within (role, rel_age)
dfc = df_z_cohort.copy()
need = {"cf_pct_z","cf60_z","ca60_z"}
if not need.issubset(dfc.columns):
    # build cohort z’s if necessary
    req = {"cf_pct","cf60","ca60","rel_age"}
    if not req.issubset(dfc.columns):
        raise KeyError(f"df_z_cohort missing {req - set(dfc.columns)} to compute cohort z’s.")
    if "role" not in dfc.columns:
        if "position" in dfc.columns:
            dfc["role"] = np.where(dfc["position"].astype(str).str.upper().str.startswith("D"), "D", "F")
        else:
            raise KeyError("df_z_cohort needs 'role' or 'position' to compute cohort z’s.")
    grp = dfc.groupby(["role","rel_age"], observed=True)
    for raw, zname in [("cf_pct","cf_pct_z"), ("cf60","cf60_z"), ("ca60","ca60_z")]:
        mu = grp[raw].transform("mean")
        sd = grp[raw].transform("std").replace(0, np.nan)
        dfc[zname] = (dfc[raw] - mu) / sd

dfz_cohort_cmp = add_composites(dfc)

# ===== SELF stack =====
m_unw_self = mean_ci(dfz_self, "z_comp_unweighted")
m_w_self   = mean_ci(dfz_self, "z_comp_weighted")
ribbon_line(m_unw_self, "Composite z (Unweighted) — Mean by rel_age (95% CI) [SELF]", "unweighted z").show()
ribbon_line(m_w_self,   "Composite z (Role-weighted) — Mean by rel_age (95% CI) [SELF]", "role-weighted z").show()

import plotly.express as px
px.histogram(dfz_self, x="z_comp_unweighted", color="role", barmode="overlay", opacity=0.6,
             title="Composite z (Unweighted) — Distribution by Role [SELF]").show()
px.histogram(dfz_self, x="z_comp_weighted", color="role", barmode="overlay", opacity=0.6,
             title="Composite z (Role-weighted) — Distribution by Role [SELF]").show()
px.box(dfz_self, x="role", y="z_comp_unweighted",
       title="Composite z (Unweighted) — Boxplot by Role [SELF]").show()
px.box(dfz_self, x="role", y="z_comp_weighted",
       title="Composite z (Role-weighted) — Boxplot by Role [SELF]").show()

# ===== COHORT stack (same visuals, labeled) =====
m_unw_coh = mean_ci(dfz_cohort_cmp, "z_comp_unweighted")
m_w_coh   = mean_ci(dfz_cohort_cmp, "z_comp_weighted")
ribbon_line(m_unw_coh, "Composite z (Unweighted) — Mean by rel_age (95% CI) [COHORT]", "unweighted z").show()
ribbon_line(m_w_coh,   "Composite z (Role-weighted) — Mean by rel_age (95% CI) [COHORT]", "role-weighted z").show()

px.histogram(dfz_cohort_cmp, x="z_comp_unweighted", color="role", barmode="overlay", opacity=0.6,
             title="Composite z (Unweighted) — Distribution by Role [COHORT]").show()
px.histogram(dfz_cohort_cmp, x="z_comp_weighted", color="role", barmode="overlay", opacity=0.6,
             title="Composite z (Role-weighted) — Distribution by Role [COHORT]").show()
px.box(dfz_cohort_cmp, x="role", y="z_comp_unweighted",
       title="Composite z (Unweighted) — Boxplot by Role [COHORT]").show()
px.box(dfz_cohort_cmp, x="role", y="z_comp_weighted",
       title="Composite z (Role-weighted) — Boxplot by Role [COHORT]").show()


In [11]:
# ============================================
# Unified plotting API for SELF / COHORT / EXCESS
# ============================================
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROLE_COLORS = {"F": "#ff7f0e", "D": "#1f77b4"}
REL_ORDER   = [-2, -1, 0, 1, 2]

def _chk(df, name):
    need = {"role","rel_age","z_comp_unweighted","z_comp_weighted"}
    missing = need - set(df.columns)
    print(f"[{name}] rows={len(df)}  missing={missing}")
    print(df[["role","rel_age"]].head(3))
_chk(dfz_self,        "SELF")
_chk(dfz_cohort_cmp,  "COHORT")

# add this small helper once
def _hex_rgba(hex_color: str, alpha: float = 0.2) -> str:
    h = hex_color.lstrip("#")
    if len(h) != 6:
        raise ValueError(f"Expected 6-digit hex like '#RRGGBB', got {hex_color}")
    r = int(h[0:2], 16); g = int(h[2:4], 16); b = int(h[4:6], 16)
    return f"rgba({r},{g},{b},{alpha})"

def _order_rel_age(df):
    if "rel_age" in df.columns:
        df = df.copy()
        df["rel_age"] = pd.Categorical(pd.to_numeric(df["rel_age"], errors="coerce"),
                                       categories=REL_ORDER, ordered=True)
    return df

def _pick_col(metric: str, df=None) -> str:
    m = (metric or "").strip().lower()
    col = {"unweighted":"z_comp_unweighted",
           "unw":"z_comp_unweighted",
           "u":"z_comp_unweighted",
           "weighted":"z_comp_weighted",
           "wtd":"z_comp_weighted",
           "w":"z_comp_weighted"}.get(m)
    if col is None:
        raise ValueError("metric must be 'unweighted' or 'weighted'")
    if df is not None and col not in df.columns:
        raise ValueError(f"column '{col}' not found in dataframe")
    return col


def _mean_ci(df, col):
    g = df.groupby(["role","rel_age"], observed=True)[col]
    out = g.agg(n="count", mean="mean", sd="std").reset_index()
    out["sem"]   = out["sd"] / out["n"].clip(lower=1)**0.5
    out["ci_lo"] = out["mean"] - 1.96*out["sem"]
    out["ci_hi"] = out["mean"] + 1.96*out["sem"]
    return _order_rel_age(out)

def _spread(df, col):
    g = df.groupby(["role","rel_age"], observed=True)[col]
    out = g.agg(sd="std",
                iqr=lambda s: (s.quantile(0.75) - s.quantile(0.25))).reset_index()
    return _order_rel_age(out)

def _tails(df, col, thr, ge=True):
    g = df.groupby(["role","rel_age"], observed=True)[col]
    if ge:
        out = g.apply(lambda s: float((s >= thr).mean())).reset_index(name=f"prop_ge_{thr}")
    else:
        out = g.apply(lambda s: float((s <= thr).mean())).reset_index(name=f"prop_le_{thr}")
    return _order_rel_age(out)


def _quantiles(df, col):
    g = df.groupby(["role","rel_age"], observed=True)[col]
    out = g.quantile([0.25,0.5,0.75]).reset_index()
    # out has columns: role, rel_age, level_2 (quantile), col
    if out.empty:
        return pd.DataFrame(columns=["role","rel_age","q25","q50","q75"])
    qcol = out.columns[-1]
    out = out.pivot(index=["role","rel_age"], columns=out.columns[-2], values=qcol).reset_index()
    # rename whatever quantiles exist
    rename_map = {}
    if 0.25 in out.columns: rename_map[0.25] = "q25"
    if 0.50 in out.columns: rename_map[0.50] = "q50"
    if 0.75 in out.columns: rename_map[0.75] = "q75"
    out = out.rename(columns=rename_map)
    # ensure all expected columns present
    for k in ("q25","q50","q75"):
        if k not in out.columns: out[k] = np.nan
    return _order_rel_age(out[["role","rel_age","q25","q50","q75"]])


# replace your _line_mean_ci with this version
def _line_mean_ci(summary_df, title, ylab):
    s = summary_df.copy()
    s["rel_age"] = s["rel_age"].astype(str)
    fig = go.Figure()
    for r in ["F","D"]:
        sub = s[s["role"]==r]
        if sub.empty: continue
        c = ROLE_COLORS[r]
        fig.add_trace(go.Scatter(x=sub["rel_age"], y=sub["ci_hi"], mode="lines",
                                 line=dict(width=0), showlegend=False, hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=sub["rel_age"], y=sub["ci_lo"], mode="lines",
                                 line=dict(width=0), fill="tonexty",
                                 fillcolor=_hex_rgba(c, 0.2),
                                 showlegend=False, hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=sub["rel_age"], y=sub["mean"], mode="lines+markers",
                                 name=f"{r}", line=dict(color=c), marker=dict(color=c)))
    fig.update_layout(title=title, template="plotly_white",
                      xaxis_title="rel_age", yaxis_title=ylab, legend_title="Role")
    fig.update_xaxes(categoryorder="array", categoryarray=[str(x) for x in REL_ORDER])
    return fig


def _line_simple(df, ycol, title, ylab):
    s = df.copy()
    s["rel_age"] = s["rel_age"].astype(str)
    fig = px.line(s, x="rel_age", y=ycol, color="role", title=title,
                  color_discrete_map=ROLE_COLORS, markers=True,
                  labels={ycol: ylab})
    fig.update_layout(template="plotly_white", legend_title="Role")
    fig.update_xaxes(categoryorder="array", categoryarray=[str(x) for x in REL_ORDER])
    return fig


# and if you use the quantile ribbon helper, patch its fill too
def _quantile_ribbon(summary_df, title, ylab):
    s = summary_df.copy()
    s["rel_age"] = s["rel_age"].astype(str)
    fig = go.Figure()
    for r in ["F","D"]:
        sub = s[s["role"]==r]
        if sub.empty: continue
        c = ROLE_COLORS[r]
        fig.add_trace(go.Scatter(x=sub["rel_age"], y=sub["q75"], mode="lines",
                                 line=dict(width=0), showlegend=False, hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=sub["rel_age"], y=sub["q25"], mode="lines",
                                 line=dict(width=0), fill="tonexty",
                                 fillcolor=_hex_rgba(c, 0.2),
                                 showlegend=False, hoverinfo="skip"))
        fig.add_trace(go.Scatter(x=sub["rel_age"], y=sub["q50"], mode="lines+markers",
                                 name=f"{r} median", line=dict(color=c), marker=dict(color=c)))
    fig.update_layout(title=title, template="plotly_white",
                      xaxis_title="rel_age", yaxis_title=ylab, legend_title="")
    fig.update_xaxes(categoryorder="array", categoryarray=[str(x) for x in REL_ORDER])
    return fig


def build_excess(df_self, df_cohort, col, align_keys=("player","rel_age","role")):
    # add a season key if both have it to avoid accidental cross-joins
    extra = None
    for k in ("season_end","season","peak_year","start_year"):
        if k in df_self.columns and k in df_cohort.columns:
            extra = k; break
    keys = list(align_keys) + ([extra] if extra else [])
    m = (df_self[keys + [col]].merge(df_cohort[keys + [col]],
         on=keys, suffixes=("_self","_cohort")))
    m["excess"] = m[f"{col}_self"] - m[f"{col}_cohort"]
    return m

def plot_view(dfz_self, dfz_cohort_cmp, *, mode: str,
              metric: str = "unweighted",
              tail_thr: float = 1.0):
    col = _pick_col(metric)  # don’t validate yet
    mode = (mode or "").lower()

    if mode in ("mean","sd","iqr","tail_high","tail_low","quantiles"):
        df = dfz_cohort_cmp.copy()
        col = _pick_col(metric, df)  # validate presence here

        if mode == "mean":
            s = _mean_ci(df, col)
            return _line_mean_ci(s, f"{metric.title()} — mean ±95% CI [COHORT]",
                                 f"{metric} z (cohort-std)")

        if mode in ("sd","iqr"):
            s = _spread(df, col)
            ycol = "sd" if mode=="sd" else "iqr"
            ttl  = f"{metric.title()} — dispersion ({ycol.upper()}) by rel_age [COHORT]"
            return _line_simple(s, ycol, ttl, f"{ycol.upper()} of z")

        if mode == "tail_high":
            s = _tails(df, col, thr=tail_thr, ge=True)
            return _line_simple(s, f"prop_ge_{tail_thr}",
                                f"{metric.title()} — share ≥ {tail_thr} by rel_age [COHORT]",
                                "proportion")

        if mode == "tail_low":
            s = _tails(df, col, thr=-tail_thr, ge=False)
            return _line_simple(s, f"prop_le_{-tail_thr}",
                                f"{metric.title()} — share ≤ -{tail_thr} by rel_age [COHORT]",
                                "proportion")

        if mode == "quantiles":
            s = _quantiles(df, col)
            return _quantile_ribbon(s, f"{metric.title()} — quantiles by rel_age [COHORT]",
                                    "z (q50 with q25–q75)")

    if mode == "excess_mean":
        # Self − Cohort, then mean ± CI by rel_age
        col_self   = _pick_col(metric, dfz_self)
        col_cohort = _pick_col(metric, dfz_cohort_cmp)
        m = build_excess(dfz_self, dfz_cohort_cmp, col_self)  # col name same on both frames
        s = _mean_ci(m.rename(columns={"excess":"y"}), "y")
        return _line_mean_ci(s, f"Excess (SELF − COHORT) — {metric.title()} mean ±95% CI",
                             "Self − Cohort (z)")

    raise ValueError("Unknown mode. Use: 'mean','sd','iqr','tail_high','tail_low','quantiles','excess_mean'")


    # if mode == "excess_mean":
    #     # Self − Cohort, then mean ± CI by rel_age
    #     m = build_excess(dfz_self, dfz_cohort_cmp, col)
    #     s = _mean_ci(m.rename(columns={"excess":"y"}), "y")
    #     return _line_mean_ci(s, f"Excess (SELF − COHORT) — {metric.title()} mean ±95% CI",
    #                          "Self − Cohort (z)")

    # raise ValueError("Unknown mode. Use: 'mean','sd','iqr','tail_high','tail_low','quantiles','excess_mean'")


[SELF] rows=1410  missing=set()
  role rel_age
0    F      -1
1    F       2
2    F      -2
[COHORT] rows=1410  missing=set()
  role rel_age
0    F      -2
1    D      -2
2    F      -2


In [12]:
plot_view(dfz_self, dfz_cohort_cmp, mode="mean",        metric="unweighted").show()
plot_view(dfz_self, dfz_cohort_cmp, mode="mean",        metric="weighted").show()
plot_view(dfz_self, dfz_cohort_cmp, mode="sd",          metric="unweighted").show()
plot_view(dfz_self, dfz_cohort_cmp, mode="iqr",         metric="unweighted").show()
plot_view(dfz_self, dfz_cohort_cmp, mode="tail_high",   metric="unweighted", tail_thr=1.0).show()
plot_view(dfz_self, dfz_cohort_cmp, mode="tail_low",    metric="unweighted", tail_thr=1.0).show()
plot_view(dfz_self, dfz_cohort_cmp, mode="quantiles",   metric="weighted").show()
plot_view(dfz_self, dfz_cohort_cmp, mode="excess_mean", metric="unweighted").show()
plot_view(dfz_self, dfz_cohort_cmp, mode="excess_mean", metric="weighted").show()


# Reading the Figures: Self vs Cohort, Levels vs Spread

## Big idea
We summarize play-driving with a composite z-score built from CF% (possession), CF/60 (creation), and CA/60 (suppression; enters with a minus).  
We look at it **two ways**:

- **SELF**: standardized within a player’s own 5-year window (t−2..t+2).  
- **COHORT**: standardized within peers at the **same role (F/D)** and **same rel_age**.

Because **cohort** z-scores are mean-centered by construction (≈ 0 at each rel_age), level plots can look “flat.”  
So we complement level with **spread** and **tails** to see *how distributions behave* around the peak arc.

---

## Panels & what they mean

### 1) Mean ± 95% CI (COHORT)
- **What it shows:** Average composite z by rel_age for D vs F, with 95% CIs.
- **Why it hovers near 0:** Cohort z-scores are standardized within each `(role, rel_age)` group, so their mean is ~0 by design.
- **How to read it:** Use it as a **sanity check** (no large systematic bias). The story here comes from the next panels.

---

### 2) Dispersion (SD / IQR) by rel_age (COHORT)
- **What it shows:** How **wide** the cohort distribution is at each rel_age.
  - **SD**: sensitive to outliers; captures overall spread.
  - **IQR**: robust middle-50% spread.
- **How to read it:** Higher spread ⇒ more variability in outcomes at that rel_age (more “boom/bust” potential).  
Compare D vs F to see where roles diverge in consistency around peaks.

---

### 3) Tail proportions (≥ +1 z and ≤ −1 z) (COHORT)
- **What it shows:** The **share of players** who are clearly **above** (≥ +1) or **below** (≤ −1) cohort at each rel_age.
- **How to read it:** Peaks in the ≥ +1 line indicate rel_age points where **standout seasons** are most frequent.  
Likewise, rises in ≤ −1 indicate more **under-performance** relative to peers.

---

### 4) Quantile ribbons (median with IQR) (COHORT)
- **What it shows:** Median (q50) line with interquartile (q25–q75) ribbon by rel_age.
- **How to read it:**  
  - **Median shift** ⇒ typical player moves up/down relative to cohort.  
  - **Ribbon width** ⇒ how tightly/loosely most players cluster.

---

### 5) Excess (SELF − COHORT) mean ± 95% CI
- **What it shows:** For each player and rel_age, the gap between their **self-standardized** score and their **cohort-standardized** score, averaged by group with CIs.
- **How to read it:** Positive excess ⇒ players tend to sit **above** their cohort baseline given their own historical arc; negative ⇒ **below**.  
This has an interpretable center (not pinned to 0) and blends the two perspectives.

---

## Practical pointers
- **Use SELF** to track individual arcs (how much a player outkicks *their own* history at each rel_age).
- **Use COHORT** to understand rarity/dispersion vs peers (how exceptional or common those arcs are at the same stage).
- **Combine with Excess** to ask: “Are players typically above peers when they’re above themselves?”

---

# How to use the plotting API

You have a single function:

```python
plot_view(dfz_self, dfz_cohort_cmp, mode=..., metric=..., tail_thr=...)


In [13]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

# === Prepare frames ===
# (You already have these from earlier steps)
# dfz_self: self-standardized, dfz_cohort_cmp: cohort-standardized
# Build EXCESS merge:
cols = ["player","rel_age","role"]
season_key = next((k for k in ["season_end","season","peak_year","start_year"]
                   if k in dfz_self.columns and k in dfz_cohort_cmp.columns), None)
keys = cols + ([season_key] if season_key else [])

m = (dfz_self[keys + ["z_comp_unweighted","z_comp_weighted"]]
     .merge(dfz_cohort_cmp[keys + ["z_comp_unweighted","z_comp_weighted"]],
            on=keys, suffixes=("_self","_cohort")))
m["excess_unw"] = m["z_comp_unweighted_self"] - m["z_comp_unweighted_cohort"]
m["excess_wtd"] = m["z_comp_weighted_self"]   - m["z_comp_weighted_cohort"]

# Ensure rel_age categorical baseline at 0:
order = [-2,-1,0,1,2]
for df in (m, dfz_cohort_cmp):
    df["rel_age"] = pd.Categorical(pd.to_numeric(df["rel_age"], errors="coerce"),
                                   categories=order, ordered=True)

# === A) OLS: Excess ~ role * rel_age (HC3 robust) ===
mod_ex = smf.ols("excess_unw ~ C(role) * C(rel_age, Treatment(0))", data=m).fit(cov_type="HC3")
print(mod_ex.summary())

# === B) GLM Binomial: Tail ≥ 1.0 vs role * rel_age, cluster by player ===
d = dfz_cohort_cmp.copy()
d["hi"] = (d["z_comp_unweighted"] >= 1.0).astype(int)
mod_tail = smf.glm("hi ~ C(role) * C(rel_age, Treatment(0))",
                   data=d, family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": d["player"]})
print(mod_tail.summary())

# === C) OLS: |z| ~ role * rel_age (HC3 robust) ===
d["abs_z"] = d["z_comp_unweighted"].abs()
mod_abs = smf.ols("abs_z ~ C(role) * C(rel_age, Treatment(0))",
                  data=d).fit(cov_type="HC3")
print(mod_abs.summary())


                            OLS Regression Results                            
Dep. Variable:             excess_unw   R-squared:                       0.015
Model:                            OLS   Adj. R-squared:                  0.009
Method:                 Least Squares   F-statistic:                     2.377
Date:                Thu, 02 Oct 2025   Prob (F-statistic):             0.0114
Time:                        00:48:22   Log-Likelihood:                -1436.9
No. Observations:                1410   AIC:                             2894.
Df Residuals:                    1400   BIC:                             2946.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

# Statistical Tests & How to Read Them

This section documents the hypothesis tests we run in **Part B** and how to interpret their output in the `statsmodels` summaries.

---

## A) OLS — **Excess (SELF − COHORT)**

**Goal.** Test whether players sit above/below their cohort baseline across the peak arc, and whether this differs by role (F/D) and `rel_age`.

**Outcome (y):**  
- `excess_unw` or `excess_wtd` = `z_comp_*_self − z_comp_*_cohort`

**Model (with rel_age baseline at 0):**  
- `y ~ C(role) * C(rel_age, Treatment(0))`

**Key tests & fields in the summary**
- **Prob (F-statistic):** overall model p-value (are all slopes jointly zero?).  
  - **H₀:** all slope coefficients = 0 (no explanatory power).  
  - **H₁:** at least one slope ≠ 0.
- **coef / std err / t / P>|t| for each term:** individual parameter tests.  
  - **H₀:** coefficient = 0.  
  - **Interpretation:** sign shows direction vs the baseline; size is the effect in **z units**.
- **[0.025, 0.975] CI:** 95% confidence interval for each coefficient.
- **R² / Adj. R²:** fraction of variance explained (fit quality; not a test).

**Robustness**
- We recommend **HC3** robust SEs (`fit(cov_type="HC3")`).
- Optional: **cluster by player** if you keep repeated measures per player.

**Report language (example)**
> *Excess (unweighted) shows a significant overall fit (F, p < 0.001). Forwards at `rel_age = +1` are +0.12 z above the baseline (t = 3.1, p = 0.002), while Defense differences at the same `rel_age` are not statistically different from zero (p = 0.21).*

---

## B) GLM Binomial — **Tail Probability vs Cohort**

**Goal.** How often do players clear a “standout” bar relative to peers at each `rel_age`?

**Outcome (y):**  
- Indicator `hi = 1{ z_comp_* ≥ τ }` with τ ∈ {1.0, 1.5, 2.0}

**Model:**  
- `hi ~ C(role) * C(rel_age, Treatment(0))` (GLM family = Binomial, link = logit)

**Key tests & fields in the summary**
- **coef / std err / z / P>|z|:** logistic coefficients (log-odds scale).  
  - **H₀:** coefficient = 0 (no change in log-odds vs baseline).  
  - **Interpretation:** exponentiate to get **odds ratios**: `OR = exp(coef)`.
- **LLR p-value:** model-level likelihood-ratio test vs the intercept-only model.
- **AIC / BIC:** information criteria (lower is better, for model comparison).

**Robustness**
- Use **clustered SEs by player** (`fit(cov_type="cluster", cov_kwds={"groups": player_id})`) to handle within-player dependence.

**Report language (example)**
> *At `rel_age = 0`, Forwards are more likely to be ≥ +1 z than baseline (OR = 1.35, z = 2.6, p = 0.009), while Defense is not (p = 0.18). Model LLR p < 0.001.*

---

## C) OLS — **Magnitude (How Extreme vs Cohort)**

**Goal.** Are players more or less **extreme** relative to peers at different `rel_age` (and by role)?

**Outcome (y):**  
- `abs_z = | z_comp_unweighted |` (or weighted)

**Model:**  
- `abs_z ~ C(role) * C(rel_age, Treatment(0))`

**Key tests**
- **Prob (F-statistic):** overall significance (as in A).  
- **Per-coefficient t-tests:** where magnitude differs from baseline (0).

**Interpretation**
- Positive coefficients: more extreme performances vs peers at that `rel_age` / role.

**Report language (example)**
> *Absolute z is higher at `rel_age = +1` for Forwards (+0.08, t = 2.2, p = 0.028), indicating more extreme outcomes vs peers near the peak.*

---

## D) Why we **don’t** regress the cohort mean level

Cohort-standardized scores are **mean-centered at 0** within `(role, rel_age)`. An OLS on levels will trivially recover intercepts ~0 with little insight. Use tests **A–C** instead (excess, tails, magnitude).

---

## Model-building checklist

1. **Design matrix**
   - Treat `rel_age` as **categorical** with baseline at 0: `C(rel_age, Treatment(0))`.
   - Include **role × rel_age** interaction if you want separate arcs by role.

2. **Variance & dependence**
   - Prefer **HC3** robust SEs.  
   - Cluster by `player` when responses include repeats per player (GLM tails especially).

3. **Multiple testing**
   - If making many pairwise statements, consider FDR control or keep focus on a few pre-registered contrasts (e.g., `rel_age ∈ {−1, 0, +1}`).

4. **Effect size reporting**
   - OLS: report **estimate (z units)** with **95% CI** and **p-value**.  
   - GLM: report **odds ratios** (`exp(coef)`) with **95% CI** and **p-value**.
   - Always mention the **baseline** (role, `rel_age=0`).

---

## Minimal code patterns

### A) OLS Excess (HC3)
```python
mod_ex = smf.ols("excess_unw ~ C(role) * C(rel_age, Treatment(0))",
                 data=m).fit(cov_type="HC3")
print(mod_ex.summary())


In [14]:
#GLM Tail clustered by player
d = dfz_cohort_cmp.copy()
d["hi"] = (d["z_comp_unweighted"] >= 1.0).astype(int)
mod_tail = smf.glm("hi ~ C(role) * C(rel_age, Treatment(0))",
                   data=d, family=sm.families.Binomial()).fit(
    cov_type="cluster", cov_kwds={"groups": d["player"]})
print(mod_tail.summary())


                 Generalized Linear Model Regression Results                  
Dep. Variable:                     hi   No. Observations:                 1410
Model:                            GLM   Df Residuals:                     1400
Model Family:                Binomial   Df Model:                            9
Link Function:                  Logit   Scale:                          1.0000
Method:                          IRLS   Log-Likelihood:                -522.76
Date:                Thu, 02 Oct 2025   Deviance:                       1045.5
Time:                        00:48:22   Pearson chi2:                 1.41e+03
No. Iterations:                     5   Pseudo R-squ. (CS):           0.005784
Covariance Type:              cluster                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

In [15]:
#OLS Magnitude HC3
d = dfz_cohort_cmp.copy()
d["abs_z"] = d["z_comp_unweighted"].abs()
mod_abs = smf.ols("abs_z ~ C(role) * C(rel_age, Treatment(0))",
                  data=d).fit(cov_type="HC3")
print(mod_abs.summary())


                            OLS Regression Results                            
Dep. Variable:                  abs_z   R-squared:                       0.003
Model:                            OLS   Adj. R-squared:                 -0.003
Method:                 Least Squares   F-statistic:                    0.5293
Date:                Thu, 02 Oct 2025   Prob (F-statistic):              0.854
Time:                        00:48:22   Log-Likelihood:                -1130.5
No. Observations:                1410   AIC:                             2281.
Df Residuals:                    1400   BIC:                             2334.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

How to reference the overall model p-value
In OLS summaries:
“Prob (F-statistic)” is the model-level p-value (overall F-test).
Coefficient P>|t| values are for individual terms.
In GLM summaries:
Look at the LLR p-value (likelihood-ratio test vs intercept-only).
One-paragraph template for your report
We modeled the excess composite (SELF−COHORT) with OLS using role × rel_age fixed effects and HC3 robust SEs. The model was significant (F, p = …). Relative to rel_age = 0, Forwards showed +… z at rel_age = +1 (95% CI […, …], p = …), while Defense did not differ materially (p = …). Complementary GLM models of tail probability (≥ +1 z) with player-clustered SEs showed higher odds at rel_age = +1 for Forwards (OR = …, p = …). Absolute z analyses indicated greater extremity at rel_age = … for ….

## Part B — Spicy (within-player z)

**What is it?**
A composite built from within-player standardized components; it answers:  
**“Relative to this same player’s five-year baseline, how hot/cold is this season?”**

This section uses the **z-table** we built earlier (`player_five_year_aligned_z`).  
We’ll explore **spicy (unweighted)** on its own.


In [16]:
z = df_z.copy()

# Distribution by role
fig = px.histogram(
    z,
    x="spicy_score",
    color="role",
    barmode="overlay",
    nbins=40,
    opacity=0.6,
    title="Spicy (z) — Distribution by Role",
)
fig.show()

# Mean spicy by rel_age × role
if "rel_age" in z.columns:
    mean_spicy = z.groupby(["role", "rel_age"], as_index=False)["spicy_score"].mean(
        numeric_only=True
    )
    fig2 = px.line(
        mean_spicy,
        x="rel_age",
        y="spicy_score",
        color="role",
        markers=True,
        title="Spicy (z) — Mean by Role × rel_age",
    )
    fig2.update_xaxes(categoryorder="array", categoryarray=[-2, -1, 0, 1, 2])
    fig2.show()

z[["player", "season", "role", "rel_age", "spicy_score"]].head(8)

/var/folders/hb/kbyr_0y166n0nvnd_cqyjzth0000gn/T/ipykernel_76124/2095448782.py:17: FutureWarning:

The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.



,player,season,role,rel_age,spicy_score
0,Adam Henrique,16-17,F,-1,-0.405994
1,Adam Henrique,19-20,F,2,0.778992
2,Adam Henrique,15-16,F,-2,-0.232633
3,Adam Henrique,17-18,F,0,0.520218
4,Adam Henrique,18-19,F,1,-0.660584
5,Adam Larsson,20-21,D,0,-0.557387
6,Adam Larsson,22-23,D,2,1.471390
7,Adam Larsson,21-22,D,1,0.027213


In [17]:
# OLS: spicy ~ role * rel_age (categorical), HC3 robust SEs
if "rel_age" in z.columns:
    m_spicy = smf.ols("spicy_score ~ C(role) * C(rel_age, Treatment(0))", data=z).fit(
        cov_type="HC3"
    )
else:
    m_spicy = smf.ols("spicy_score ~ C(role)", data=z).fit(cov_type="HC3")

print(m_spicy.summary())

# Tidy
coefs_spicy = pd.DataFrame(
    {
        "term": m_spicy.params.index,
        "coef": m_spicy.params.values,
        "se": m_spicy.bse.values,
        "p": m_spicy.pvalues.values,
    }
)
ci = m_spicy.conf_int()
coefs_spicy["ci_lo"] = ci[0].values
coefs_spicy["ci_hi"] = ci[1].values
coefs_spicy

                            OLS Regression Results                            
Dep. Variable:            spicy_score   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     1.816
Date:                Thu, 02 Oct 2025   Prob (F-statistic):             0.0610
Time:                        00:48:22   Log-Likelihood:                -1588.3
No. Observations:                1410   AIC:                             3197.
Df Residuals:                    1400   BIC:                             3249.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

,term,coef,se,p,ci_lo,ci_hi
0,Intercept,-0.009147,0.067207,0.891746,-0.140870,0.122577
1,C(role)[T.F],0.009825,0.087905,0.911010,-0.162466,0.182115
2,"C(rel_age, Treatment(0))[T.-2]",0.075788,0.108040,0.483002,-0.135966,0.287542
3,"C(rel_age, Treatment(0))[T.-1]",0.069372,0.096981,0.474413,-0.120707,0.259452
4,"C(rel_age, Treatment(0))[T.1]",-0.008445,0.103823,0.935173,-0.211934,0.195044
5,"C(rel_age, Treatment(0))[T.2]",-0.090983,0.100895,0.367183,-0.288733,0.106767
6,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-2]",-0.068091,0.135083,0.614213,-0.332848,0.196666
7,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-1]",0.053009,0.123868,0.668686,-0.189767,0.295786
8,"C(role)[T.F]:C(rel_age, Treatment(0))[T.1]",0.039235,0.128253,0.759668,-0.212137,0.290606
9,"C(role)[T.F]:C(rel_age, Treatment(0))[T.2]",-0.073276,0.130671,0.574955,-0.329387,0.182834


## Part C — Spicy-Weighted (within-player z, role-aware)

This is the role-aware composite **in z-space** (already computed in SQL as `spicy_weighted`).  
Interpretation remains within-player: positive = hotter than that player’s own baseline, negative = colder.


In [18]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

# --- start with your table ---
zw = dfz.copy()

# ensure role as D/F
role_col = "position" if "position" in zw.columns else "role"
zw["role"] = zw[role_col].astype(str).str.upper().str.startswith("D").map({True: "D", False: "F"})

# keep useful columns if they exist (DON'T drop others)
keep_maybe = ["spicy_score", "spicy_weighted", "role", "rel_age", "player", "season"]
zw = zw[[c for c in keep_maybe if c in zw.columns]].copy()

# -------- A) SCATTER: spicy (x) vs spicy_weighted (y), faceted by role --------
sc = zw.dropna(
    subset=[c for c in ["spicy_score", "spicy_weighted", "role"] if c in zw.columns]
).copy()

mn = float(np.nanmin([sc["spicy_score"].min(), sc["spicy_weighted"].min()]))
mx = float(np.nanmax([sc["spicy_score"].max(), sc["spicy_weighted"].max()]))
pad = (mx - mn) * 0.05 if np.isfinite(mx - mn) else 0.1
rng = [mn - pad, mx + pad]

fig = px.scatter(
    sc,
    x="spicy_score",
    y="spicy_weighted",
    facet_row="role",
    color="role",
    opacity=0.6,
    trendline="ols",
    trendline_scope="trace",
    labels={"spicy_score": "SPICY (equal-weight)", "spicy_weighted": "SPICY (role-weighted)"},
    title="Within-role comparison: SPICY vs SPICY (role-weighted)",
)

# add y=x line to each facet + shared axes
for r in ["D", "F"]:
    fig.add_trace(
        go.Scatter(
            x=rng,
            y=rng,
            mode="lines",
            line=dict(width=1, dash="dash"),
            name="y = x",
            showlegend=(r == "D"),
        ),
        row=1 if r == "D" else 2,
        col=1,
    )
fig.update_xaxes(matches="x", range=rng, tickformat=".2f")
fig.update_yaxes(matches="y", range=rng, tickformat=".2f")
fig.for_each_annotation(lambda a: a.update(text=a.text.replace("role=", "")))
fig.update_layout(
    template="plotly_white", legend_title_text="Role", margin=dict(l=60, r=10, t=60, b=40)
)
fig.show()

# -------- B) LINES: mean spicy_weighted by rel_age × role (only if rel_age exists) --------
if "rel_age" in zw.columns:
    # normalize rel_age to ordered set
    order = [-2, -1, 0, 1, 2]
    zw["rel_age"] = pd.to_numeric(zw["rel_age"], errors="coerce")
    zw = zw[zw["rel_age"].isin(order)].copy()
    zw["rel_age"] = pd.Categorical(zw["rel_age"], categories=order, ordered=True)

    g = (
        zw.groupby(["role", "rel_age"], observed=True)["spicy_weighted"]
        .mean(numeric_only=True)
        .reset_index()
    )
    fig_lines = px.line(
        g,
        x="rel_age",
        y="spicy_weighted",
        color="role",
        markers=True,
        title="Spicy-Weighted (z) — Mean by Role × rel_age",
        category_orders={"rel_age": order, "role": ["D", "F"]},
    )
    fig_lines.update_xaxes(categoryorder="array", categoryarray=order)
    fig_lines.update_yaxes(tickformat=".2f", dtick=0.05)
    fig_lines.update_layout(template="plotly_white")
    fig_lines.show()

# -------- C) Optional: quick peek of columns IF present (no KeyError) --------
peek_cols = [
    c for c in ["player", "season", "role", "rel_age", "spicy_weighted"] if c in zw.columns
]
if peek_cols:
    print(zw[peek_cols].head(8).to_string(index=False))
else:
    print("(Peek skipped: none of player/season/rel_age present.)")

       player season role rel_age  spicy_weighted
Adam Henrique  16-17    F      -1       -0.428285
Adam Henrique  19-20    F       2        0.931875
Adam Henrique  15-16    F      -2       -0.538525
Adam Henrique  17-18    F       0        0.683384
Adam Henrique  18-19    F       1       -0.648449
 Adam Larsson  20-21    D       0       -0.552449
 Adam Larsson  22-23    D       2        1.507333
 Adam Larsson  21-22    D       1        0.044183


In [19]:
# OLS: spicy_weighted ~ role * rel_age (categorical), HC3 robust SEs
if "rel_age" in zw.columns:
    m_sw = smf.ols("spicy_weighted ~ C(role) * C(rel_age, Treatment(0))", data=zw).fit(
        cov_type="HC3"
    )
else:
    m_sw = smf.ols("spicy_weighted ~ C(role)", data=zw).fit(cov_type="HC3")

print(m_sw.summary())

# Tidy
coefs_sw = pd.DataFrame(
    {
        "term": m_sw.params.index,
        "coef": m_sw.params.values,
        "se": m_sw.bse.values,
        "p": m_sw.pvalues.values,
    }
)
ci = m_sw.conf_int()
coefs_sw["ci_lo"] = ci[0].values
coefs_sw["ci_hi"] = ci[1].values
coefs_sw

                            OLS Regression Results                            
Dep. Variable:         spicy_weighted   R-squared:                       0.012
Model:                            OLS   Adj. R-squared:                  0.006
Method:                 Least Squares   F-statistic:                     1.835
Date:                Thu, 02 Oct 2025   Prob (F-statistic):             0.0580
Time:                        00:48:22   Log-Likelihood:                -1652.5
No. Observations:                1410   AIC:                             3325.
Df Residuals:                    1400   BIC:                             3377.
Df Model:                           9                                         
Covariance Type:                  HC3                                         
                                                  coef    std err          z      P>|z|      [0.025      0.975]
--------------------------------------------------------------------------------------------------

,term,coef,se,p,ci_lo,ci_hi
0,Intercept,-0.013688,0.070520,0.846097,-0.151905,0.124529
1,C(role)[T.F],0.004032,0.091895,0.965007,-0.176079,0.184142
2,"C(rel_age, Treatment(0))[T.-2]",0.088182,0.112975,0.435070,-0.133245,0.309609
3,"C(rel_age, Treatment(0))[T.-1]",0.088050,0.100957,0.383129,-0.109823,0.285923
4,"C(rel_age, Treatment(0))[T.1]",-0.013579,0.108420,0.900331,-0.226078,0.198920
5,"C(rel_age, Treatment(0))[T.2]",-0.094213,0.105886,0.373599,-0.301746,0.113321
6,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-2]",-0.072322,0.141202,0.608519,-0.349074,0.204429
7,"C(role)[T.F]:C(rel_age, Treatment(0))[T.-1]",0.050575,0.129088,0.695216,-0.202432,0.303582
8,"C(role)[T.F]:C(rel_age, Treatment(0))[T.1]",0.062279,0.134117,0.642388,-0.200585,0.325143
9,"C(role)[T.F]:C(rel_age, Treatment(0))[T.2]",-0.060689,0.136862,0.657452,-0.328934,0.207555


## Headline Numbers  This is the News! 
**Out of our sample, how many players show a peak in performance at age 27?**


In [20]:
from pathlib import Path

import numpy as np
import pandas as pd

# ---- Inputs & checks ----
try:
    dfz = df_z.copy()
except NameError as e:
    raise NameError("df_z is not defined. Load the z-table first (see earlier cell).") from e

required_cols = {"player", "peak_year", "role", "rel_age", "cf60_dz", "ca60_dz", "cf_pct_dz"}
missing = required_cols - set(dfz.columns)
if missing:
    raise ValueError(f"Missing required columns in df_z: {sorted(missing)}")

# normalize role to {"D","F"}
dfz["role"] = np.where(dfz["role"].astype(str).str.upper().str.startswith("D"), "D", "F")
# enforce rel_age numeric and filter
VALID_RELS = [-2, -1, 0, 1, 2]
dfz["rel_age"] = pd.to_numeric(dfz["rel_age"], errors="coerce")
dfz = dfz[dfz["rel_age"].isin(VALID_RELS)].copy()


# ---- Part 1: Group means + 95% CIs by role × rel_age for each delta ----
def mean_ci(df: pd.DataFrame, value_col: str) -> pd.DataFrame:
    g = (
        df.dropna(subset=[value_col])
        .groupby(["role", "rel_age"], observed=True)
        .agg(n=(value_col, "size"), mean=(value_col, "mean"), sd=(value_col, "std"))
        .reset_index()
    )
    g["se"] = g["sd"] / np.sqrt(g["n"].clip(lower=1))
    g["ci_lo"] = g["mean"] - 1.96 * g["se"]
    g["ci_hi"] = g["mean"] + 1.96 * g["se"]
    g["metric"] = value_col
    return g


means_cf60 = mean_ci(dfz, "cf60_dz")
means_ca60 = mean_ci(dfz, "ca60_dz")
means_cfpc = mean_ci(dfz, "cf_pct_dz")

means_all = pd.concat([means_cf60, means_ca60, means_cfpc], ignore_index=True)

# quick peek
print("\n=== Group means (first 10 rows) ===")
print(means_all.head(10).to_string(index=False))


# ---- Part 2: Per-player concavity test & peak location in {-1,0,1} ----
# We fit y = a*x^2 + b*x + c on the 5 points (x = rel_age), then:
#   concave peak if a < 0 and vertex x* = -b/(2a) ∈ {-1,0,1} (allow small tolerance)
def concave_peak_in_window(
    x: np.ndarray, y: np.ndarray, window=(-1, 1), tol=0.35
) -> tuple[bool, float]:
    if len(x) < 5:
        return False, np.nan
    # Fit quadratic
    a, b, c = np.polyfit(x, y, deg=2)
    if a >= 0:
        return False, -b / (2 * a)  # not concave; still return vertex for info
    x_star = -b / (2 * a)
    in_window = (x_star >= window[0] - tol) and (x_star <= window[1] + tol)
    return in_window, x_star


def classify_metric(
    df: pd.DataFrame, value_col: str, invert_for_peak: bool = False
) -> pd.DataFrame:
    # Prepare long-form per (player, peak_year)
    cols = ["player", "peak_year", "role", "rel_age", value_col]
    sub = df[cols].dropna().copy()
    # Invert if needed (ca60_dz: smaller is better → maximize -ca60_dz)
    if invert_for_peak:
        sub["_y"] = -sub[value_col]
    else:
        sub["_y"] = sub[value_col]

    records = []
    for (pl, py), g in sub.groupby(["player", "peak_year"]):
        # Require all 5 rel_age points present
        if set(g["rel_age"]) >= set(VALID_RELS):
            x = g.sort_values("rel_age")["rel_age"].to_numpy()
            y = g.sort_values("rel_age")["_y"].to_numpy()
            ok, x_star = concave_peak_in_window(x, y, window=(-1, 1), tol=0.35)
            role = g["role"].mode(dropna=False).iat[0]
            records.append(
                {
                    "player": pl,
                    "peak_year": py,
                    "role": role,
                    "metric": value_col,
                    "concave_peak_-1to1": bool(ok),
                    "vertex_rel_age": float(x_star),
                }
            )
    return pd.DataFrame.from_records(records)


cls_cf60 = classify_metric(dfz, "cf60_dz", invert_for_peak=False)  # higher near 0 is better
cls_ca60 = classify_metric(dfz, "ca60_dz", invert_for_peak=True)  # invert: lower near 0 is better
cls_cfpc = classify_metric(dfz, "cf_pct_dz", invert_for_peak=False)

class_all = pd.concat([cls_cf60, cls_ca60, cls_cfpc], ignore_index=True)


# ---- Part 3: Headline counts (by metric × role and overall) ----
def headline_counts(df: pd.DataFrame, title: str):
    print(f"\n=== {title} ===")
    # by metric × role
    by_role = (
        df.groupby(["metric", "role"], observed=True)["concave_peak_-1to1"]
        .agg(n="size", n_peak="sum")
        .reset_index()
    )
    by_role["share_peak_%"] = 100 * by_role["n_peak"] / by_role["n"].replace(0, np.nan)
    print("\nBy metric × role:")
    print(by_role.to_string(index=False, formatters={"share_peak_%": lambda v: f"{v:5.1f}"}))

    # overall by metric
    overall = (
        df.groupby(["metric"], observed=True)["concave_peak_-1to1"]
        .agg(n="size", n_peak="sum")
        .reset_index()
    )
    overall["share_peak_%"] = 100 * overall["n_peak"] / overall["n"].replace(0, np.nan)
    print("\nOverall by metric:")
    print(overall.to_string(index=False, formatters={"share_peak_%": lambda v: f"{v:5.1f}"}))
    return by_role, overall


by_role_tbl, overall_tbl = headline_counts(class_all, "Concave peak in {-1,0,1} (per metric)")

# ---- Part 4: Save summary CSVs for the paper/notebook ----
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)
means_all.to_csv(OUT / "delta_means_ci_by_role_rel_age.csv", index=False)
class_all.to_csv(OUT / "concave_peak_classification_per_player.csv", index=False)

print("\nWrote:")
print(" -", OUT / "delta_means_ci_by_role_rel_age.csv")
print(" -", OUT / "concave_peak_classification_per_player.csv")

# ---- One-line “headline” you can quote in the writeup (example for CF60 only) ----
cf60_overall = overall_tbl[overall_tbl["metric"] == "cf60_dz"].iloc[0]
print(
    f"\nHeadline (CF60_dz): {cf60_overall['n_peak']}/{cf60_overall['n']} "
    f"({cf60_overall['share_peak_%']:.1f}%) players show a concave peak "
    f"in rel_age ∈ {{-1,0,1}}."
)


=== Group means (first 10 rows) ===
role  rel_age   n      mean       sd       se     ci_lo    ci_hi  metric
   D       -2 102 -0.038329 1.333723 0.132058 -0.297163 0.220505 cf60_dz
   D       -1 102 -0.068530 1.279906 0.126730 -0.316921 0.179860 cf60_dz
   D        0 102  0.000000 0.000000 0.000000  0.000000 0.000000 cf60_dz
   D        1 102  0.044570 1.303043 0.129020 -0.208310 0.297450 cf60_dz
   D        2 102 -0.033517 1.381589 0.136798 -0.301640 0.234607 cf60_dz
   F       -2 180 -0.041192 1.399357 0.104302 -0.245624 0.163240 cf60_dz
   F       -1 180  0.127508 1.320391 0.098416 -0.065388 0.320403 cf60_dz
   F        0 180  0.000000 0.000000 0.000000  0.000000 0.000000 cf60_dz
   F        1 180  0.085375 1.352324 0.100796 -0.112186 0.282935 cf60_dz
   F        2 180 -0.045781 1.499559 0.111771 -0.264851 0.173290 cf60_dz

=== Concave peak in {-1,0,1} (per metric) ===

By metric × role:
   metric role   n  n_peak share_peak_%
  ca60_dz    D 102      34         33.3
  ca60_dz    F

## Does Weighting the score really Tilt the Ice?


In [21]:
import numpy as np
import pandas as pd

# Expect df_z already loaded
dfz = df_z.copy()

# Ensure role is D/F
dfz["role"] = np.where(dfz["role"].astype(str).str.upper().str.startswith("D"), "D", "F")

# Sanity: required base z columns
base_needed = {"cf_pct_z", "cf60_z", "ca60_z"}
missing_base = base_needed - set(dfz.columns)
if missing_base:
    raise ValueError(f"df_z is missing base z columns: {sorted(missing_base)}")

# --- Unweighted composite z: mean(cf_pct_z, cf60_z, -ca60_z)
dfz["z_comp_unweighted"] = (dfz[["cf_pct_z", "cf60_z"]].mean(axis=1) - dfz["ca60_z"] / 2.0).where(
    ~dfz[["cf_pct_z", "cf60_z", "ca60_z"]].isna().any(axis=1), np.nan
)

# Note: The above equals (cf_pct_z + cf60_z - ca60_z)/3 scaled by 1.5.
# If you prefer strict mean, use:
# dfz["z_comp_unweighted"] = (dfz["cf_pct_z"] + dfz["cf60_z"] - dfz["ca60_z"]) / 3.0

# --- Role-weighted composite z (heuristics)
# D: 0.5*cf_pct_z + 0.2*cf60_z - 0.3*ca60_z
# F: 0.5*cf_pct_z + 0.3*cf60_z - 0.2*ca60_z
w_cf60 = np.where(dfz["role"] == "D", 0.2, 0.3)
w_ca60 = np.where(dfz["role"] == "D", 0.3, 0.2)
dfz["z_comp_weighted"] = 0.5 * dfz["cf_pct_z"] + w_cf60 * dfz["cf60_z"] - w_ca60 * dfz["ca60_z"]

# Optional: if spicy_weighted not present, create it from spicy components
if "spicy_score" in dfz.columns and "spicy_weighted" not in dfz.columns:
    # Equal-weight spicy (already in spicy_score) ~ mean of cf_pct_z, cf60_z, -ca60_z
    # Weighted spicy mirrors composite weights:
    dfz["spicy_weighted"] = 0.5 * dfz["cf_pct_z"] + w_cf60 * dfz["cf60_z"] - w_ca60 * dfz["ca60_z"]

# Put back for downstream cells
df_z = dfz

print(
    "Added columns:",
    [c for c in ["z_comp_unweighted", "z_comp_weighted", "spicy_weighted"] if c in df_z.columns],
)
print(df_z[["role", "rel_age", "z_comp_unweighted", "z_comp_weighted"]].head())

Added columns: ['z_comp_unweighted', 'z_comp_weighted', 'spicy_weighted']
  role rel_age  z_comp_unweighted  z_comp_weighted
0    F      -1          -0.608991        -0.428285
1    F       2           1.168489         0.931875
2    F      -2          -0.348950        -0.538525
3    F       0           0.780327         0.683384
4    F       1          -0.990875        -0.648449


In [22]:
# === Does weighting matter? (Composite Corsi z and Spicy) ===
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

# Expect df_z already loaded. Check required columns.
need = {
    "player",
    "peak_year",
    "role",
    "rel_age",
    "z_comp_unweighted",
    "z_comp_weighted",
    "spicy_score",
    "spicy_weighted",
}
missing = need - set(df_z.columns)
if missing:
    raise ValueError(f"df_z is missing columns: {sorted(missing)}")

z = df_z.copy()
# Normalize role and rel_age
z["role"] = np.where(z["role"].astype(str).str.upper().str.startswith("D"), "D", "F")
order = [-2, -1, 0, 1, 2]
z["rel_age"] = pd.Categorical(
    pd.to_numeric(z["rel_age"], errors="coerce"), categories=order, ordered=True
)


def mean_ci(df, col, by=("role", "rel_age")):
    g = (
        df.dropna(subset=[col])
        .groupby(list(by), observed=True)
        .agg(n=(col, "size"), mean=(col, "mean"), sd=(col, "std"))
        .reset_index()
    )
    g["se"] = g["sd"] / np.sqrt(g["n"].clip(lower=1))
    g["ci_lo"] = g["mean"] - 1.96 * g["se"]
    g["ci_hi"] = g["mean"] + 1.96 * g["se"]
    g["measure"] = col
    return g


def summarize_diffs(df, weighted_col, unweighted_col, label):
    d = df[["player", "peak_year", "role", "rel_age", weighted_col, unweighted_col]].dropna().copy()
    d["diff"] = d[weighted_col] - d[unweighted_col]
    d["family"] = label
    return d


# 1) Build diff frames
d_corsi = summarize_diffs(z, "z_comp_weighted", "z_comp_unweighted", "Composite Corsi z")
d_spicy = summarize_diffs(z, "spicy_weighted", "spicy_score", "Spicy z")

diffs = pd.concat([d_corsi, d_spicy], ignore_index=True)

# 2) Summaries by role × rel_age
sum_corsi = mean_ci(d_corsi, "diff", by=("role", "rel_age"))
sum_spicy = mean_ci(d_spicy, "diff", by=("role", "rel_age"))

print("=== Mean difference (weighted − unweighted) by role × rel_age ===")
print("\nComposite Corsi z:")
print(sum_corsi.to_string(index=False, float_format=lambda v: f"{v: .3f}"))
print("\nSpicy z:")
print(sum_spicy.to_string(index=False, float_format=lambda v: f"{v: .3f}"))


# 3) Quick tests: is mean(diff) != 0 ? (overall and by role)
def mean_test_table(df, label):
    tbl = (
        df.groupby("role", observed=True)["diff"].agg(n="size", mean="mean", sd="std").reset_index()
    )
    # overall row
    overall = df["diff"].agg(n="size", mean="mean", sd="std").to_dict()
    overall["role"] = "ALL"
    tbl = pd.concat([tbl, pd.DataFrame([overall])], ignore_index=True)

    # Normal approx p-value (two-sided). For large n this is ok; else swap in scipy if desired.
    tbl["se"] = tbl["sd"] / np.sqrt(tbl["n"].clip(lower=1))
    tbl["z"] = tbl["mean"] / tbl["se"].replace(0, np.nan)
    # 95% CI
    tbl["ci_lo"] = tbl["mean"] - 1.96 * tbl["se"]
    tbl["ci_hi"] = tbl["mean"] + 1.96 * tbl["se"]

    print(f"\n=== Mean(diff) test for {label} (weighted − unweighted) ===")
    print(
        tbl[["role", "n", "mean", "se", "ci_lo", "ci_hi", "z"]].to_string(
            index=False, float_format=lambda v: f"{v: .4f}"
        )
    )
    return tbl


test_corsi = mean_test_table(d_corsi, "Composite Corsi z")
test_spicy = mean_test_table(d_spicy, "Spicy z")

# 4) Visuals — lines & histograms


# (A) Line: means of weighted vs unweighted by rel_age (separate traces) for each role
def long_two(df, a, b, label_a, label_b, family_label):
    t = (
        df[["role", "rel_age", a, b]]
        .dropna()
        .melt(id_vars=["role", "rel_age"], value_vars=[a, b], var_name="which", value_name="value")
    )
    t["which"] = t["which"].map({a: label_a, b: label_b})
    t["family"] = family_label
    return t


long_corsi = long_two(
    z, "z_comp_unweighted", "z_comp_weighted", "unweighted", "weighted", "Composite Corsi z"
)
long_spicy = long_two(z, "spicy_score", "spicy_weighted", "unweighted", "weighted", "Spicy z")

for fam, df_long in [("Composite Corsi z", long_corsi), ("Spicy z", long_spicy)]:
    fig = px.line(
        df_long,
        x="rel_age",
        y="value",
        color="which",
        facet_col="role",
        category_orders={"rel_age": order},
        markers=True,
        title=f"{fam}: weighted vs unweighted (by rel_age, faceted by role)",
        labels={"value": fam, "which": "series"},
    )
    fig.update_layout(height=380)
    fig.show()

# (B) Hist: distribution of (weighted − unweighted) by role
for fam, d in [("Composite Corsi z", d_corsi), ("Spicy z", d_spicy)]:
    fig = px.histogram(
        d,
        x="diff",
        color="role",
        barmode="overlay",
        nbins=50,
        opacity=0.55,
        title=f"{fam}: distribution of (weighted − unweighted) by role",
        labels={"diff": "weighted − unweighted"},
    )
    fig.show()


# 5) Correlation between weighted & unweighted (how similar are the rankings?)
def corr_table(df, a, b, name):
    cc = (
        df[[a, b, "role"]]
        .dropna()
        .groupby("role", observed=True)
        .apply(lambda g: pd.Series({"pearson_r": g[a].corr(g[b])}))
        .reset_index()
    )
    # overall
    overall_r = df[[a, b]].dropna().corr().iloc[0, 1]
    cc = pd.concat([cc, pd.DataFrame([{"role": "ALL", "pearson_r": overall_r}])], ignore_index=True)
    print(f"\n=== Correlation {name} (weighted vs unweighted) ===")
    print(cc.to_string(index=False, float_format=lambda v: f"{v: .3f}"))
    return cc


corr_corsi = corr_table(z, "z_comp_unweighted", "z_comp_weighted", "Composite Corsi z")
corr_spicy = corr_table(z, "spicy_score", "spicy_weighted", "Spicy z")

# 6) Save small summaries for the paper
OUT = Path("data/outputs")
OUT.mkdir(parents=True, exist_ok=True)
sum_corsi.assign(family="Composite Corsi z").to_csv(
    OUT / "weighting_effect_corsi_by_role_rel_age.csv", index=False
)
sum_spicy.assign(family="Spicy z").to_csv(
    OUT / "weighting_effect_spicy_by_role_rel_age.csv", index=False
)
test_corsi.assign(family="Composite Corsi z").to_csv(OUT / "weighting_tests_corsi.csv", index=False)
test_spicy.assign(family="Spicy z").to_csv(OUT / "weighting_tests_spicy.csv", index=False)
corr_corsi.assign(family="Composite Corsi z").to_csv(OUT / "weighting_corr_corsi.csv", index=False)
corr_spicy.assign(family="Spicy z").to_csv(OUT / "weighting_corr_spicy.csv", index=False)

print("\nSaved CSVs in data/outputs/")

=== Mean difference (weighted − unweighted) by role × rel_age ===

Composite Corsi z:
role rel_age   n   mean     sd     se  ci_lo  ci_hi measure
   D      -2 102 -0.025  0.397  0.039 -0.102  0.052    diff
   D      -1 102 -0.016  0.338  0.033 -0.081  0.050    diff
   D       0 102  0.000  0.314  0.031 -0.061  0.061    diff
   D       1 102 -0.001  0.375  0.037 -0.074  0.072    diff
   D       2 102  0.042  0.354  0.035 -0.026  0.111    diff
   F      -2 180 -0.006  0.363  0.027 -0.059  0.047    diff
   F      -1 180 -0.056  0.327  0.024 -0.103 -0.008    diff
   F       0 180 -0.011  0.359  0.027 -0.063  0.042    diff
   F       1 180 -0.008  0.305  0.023 -0.053  0.036    diff
   F       2 180  0.081  0.377  0.028  0.026  0.136    diff

Spicy z:
role rel_age   n   mean     sd     se  ci_lo  ci_hi measure
   D      -2 102  0.008  0.077  0.008 -0.007  0.023    diff
   D      -1 102  0.014  0.069  0.007  0.001  0.028    diff
   D       0 102 -0.005  0.071  0.007 -0.018  0.009    diff
   D


=== Correlation Composite Corsi z (weighted vs unweighted) ===
role  pearson_r
   D      0.996
   F      0.996
 ALL      0.996

=== Correlation Spicy z (weighted vs unweighted) ===
role  pearson_r
   D      0.996
   F      0.996
 ALL      0.996

Saved CSVs in data/outputs/


/var/folders/hb/kbyr_0y166n0nvnd_cqyjzth0000gn/T/ipykernel_76124/3297049330.py:155: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.

/var/folders/hb/kbyr_0y166n0nvnd_cqyjzth0000gn/T/ipykernel_76124/3297049330.py:155: FutureWarning:

DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.





## Weighted vs. Unweighted — What changed?

**Bottom line:** Almost nothing. Across Defense and Forwards, the weighted and unweighted composites are *virtually interchangeable*.

### Evidence

* **Correlations:** r ≈ **0.996** (D, F, and overall) between weighted and unweighted composite z’s → the two scores move together almost perfectly.
* **Mean differences (weighted − unweighted):**

  * By role × rel_age: all means are very small (mostly in **±0.06 z**), with CIs that overwhelmingly **span 0**.

    * One small blip: **Forwards @ rel_age = −1** shows −0.056 (95% CI **[−0.103, −0.008]**), i.e., a tiny downward shift when weighting—directionally consistent with giving F a bit more creation and a bit less suppression weight—but **still very small** in magnitude.
  * **Overall (pooled by role):** mean diffs are essentially **0.000** with tight CIs (e.g., ALL: 95% CI **[−0.018, 0.018]** for composite Corsi; **[−0.004, 0.004]** for spicy).

### Interpretation

* The heuristic weights (D: 0.5/0.2/−0.3 vs F: 0.5/0.3/−0.2 on cf_pct_z/cf60_z/ca60_z) **do not materially change** rankings or conclusions.
* Any “shifts” are **tiny**—on the order of **hundredths of a z-score**, i.e., practically negligible at the player-season level.
* For this milestone, it’s reasonable to present **unweighted** composite results as the primary view and note that the **weighted** variant yields the **same story**.

### Slide one-liner

> **Weighted vs. Unweighted:** No meaningful difference — r=0.996 and mean(delta)≈0 with CIs spanning 0. Same conclusions either way.

## Why the weighted band is tighter

* The **weighted composite** (role-aware z) gives **less influence** to noisier pieces (e.g., creation for D, suppression for F).
* That **reduces variance** without meaningfully shifting the center, so the **95% CI ribbon is narrower** and often **fits inside** the unweighted ribbon.
* In other words, weighting acts like a mild **regularizer**: it **stabilizes** the composite but **doesn’t change the story**—your weighted and unweighted curves are almost identical in level (correlations ≈ 1), with the weighted one simply **less wobbly**.



# Peak Δ — self vs. cohort

**Definition.**
**Peak Δ** = performance at **rel_age = 0** minus the average over **rel_age −2…+2**.

---

## Player vs **SELF** (self-relative)
How far a player’s peak rises above **their own** 5-year baseline.
**Metric:** `df_z` → self metric (`spicy_score` for equal-weight, or `spicy_weighted` if you prefer role weights).

## Player vs **COHORT** (position × rel_age)
How far a player’s peak stands out vs **peers at the same position and the same rel_age**.
**Metric:** `df_z_cohort` → cohort composite `spicy_score` *(CF% ↑, CF/60 ↑, CA/60 ↓)*.

> **Why both?**
> **Self** tells the personal arc; **Cohort** shows how exceptional the peak is in context.


In [23]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

VALID_RELS = [-2, -1, 0, 1, 2]

def normalize_roles_inplace(df: pd.DataFrame, role_col_candidates=("role","position")) -> pd.DataFrame:
    """
    Ensures df has a 'role' column with only 'F' or 'D', derived from 'role' or 'position'.
    Mutates df in place and also returns it for convenience.
    """
    for c in role_col_candidates:
        if c in df.columns:
            df["role"] = df[c].map(to_role_fd)
            return df
    raise KeyError("Need 'role' or 'position' column to derive F/D.")

def compute_peak_delta(df, value_col, group_cols=("player", "peak_year"), role_col="role"):
    """
    One row per (group_cols) containing:
      - peak_val at rel_age=0
      - win_mean over rel_age in [-2,-1,0,1,2]
      - peak_delta = peak_val - win_mean
    """
    need = set(group_cols) | {"rel_age", value_col}
    missing = need - set(df.columns)
    if missing:
        raise KeyError(f"Missing required columns for PeakΔ: {sorted(missing)}")

    dd = df.copy()
    dd["rel_age"] = pd.to_numeric(dd["rel_age"], errors="coerce")
    dd = dd[dd["rel_age"].isin(VALID_RELS)].dropna(subset=[value_col])

    pk = (dd.loc[dd["rel_age"].eq(0)]
            .groupby(list(group_cols), as_index=False)[value_col]
            .mean()
            .rename(columns={value_col: "peak_val"}))

    win = (dd.groupby(list(group_cols), as_index=False)[value_col]
             .mean()
             .rename(columns={value_col: "win_mean"}))

    out = pk.merge(win, on=list(group_cols), how="inner")
    out["peak_delta"] = out["peak_val"] - out["win_mean"]

    if role_col in df.columns:
        roles = df.groupby(list(group_cols), as_index=False)[role_col].agg(
            lambda s: s.dropna().iloc[0] if len(s.dropna()) else None
        )
        out = out.merge(roles, on=list(group_cols), how="left")

    return out


def bar_topk_peak_delta(peak_df, k=5, title="Peak Δ (Top 5) — higher = bigger lift vs norm"):
    # 1) pick top k and sort DESC so rank 1 is the largest
    topk = (peak_df.nlargest(min(k, len(peak_df)), "peak_delta")
                    .copy()
                    .sort_values("peak_delta", ascending=False)
                    .reset_index(drop=True))

    # 2) build explicit rank label
    label_core = topk["player"].astype(str) + np.where(
        "peak_year" in topk.columns, " (" + topk["peak_year"].astype(str) + ")", ""
    )
    topk["rank"]  = np.arange(1, len(topk) + 1)           # 1..k
    topk["label"] = topk["rank"].astype(str) + ". " + label_core

    # 3) collapse roles to F/D and map colors
    def to_role_fd(x):
        s = str(x).strip().upper()
        return "D" if s.startswith("D") else "F"
    role = topk.get("role", topk.get("position", "F")).map(to_role_fd)

    color_map = {"F": "#ff7f0e", "D": "#1f77b4"}
    bar_colors = role.map(color_map)

    # 4) single trace preserves y order perfectly
    y_order = topk["label"].tolist()  # 1..k in the exact order we want
    fig = go.Figure(go.Bar(
        x=topk["peak_delta"],
        y=pd.Categorical(topk["label"], categories=y_order, ordered=True),
        orientation="h",
        marker_color=bar_colors,
        hovertemplate="<b>%{y}</b><br>Peak Δ: %{x:.3f}<extra></extra>",
        showlegend=False,  # legend will come from dummy traces below
    ))

    # 5) add legend-only dummies for F and D
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=10, color=color_map["F"]),
        name="F", showlegend=True, hoverinfo="skip"
    ))
    fig.add_trace(go.Scatter(
        x=[None], y=[None], mode="markers",
        marker=dict(size=10, color=color_map["D"]),
        name="D", showlegend=True, hoverinfo="skip"
    ))

    fig.update_layout(
        title=title, template="plotly_white",
        xaxis_title="Peak Δ", yaxis_title="",
        height=320, margin=dict(l=160, r=10, t=50, b=30),
        bargap=0.2, legend_traceorder="normal"
    )
    # IMPORTANT: no axis reversing; categorical order controls display
    fig.update_yaxes(categoryorder="array", categoryarray=y_order)

    return fig


def hist_peak_delta(peak_df, title="Peak Δ — distribution by role"):
    # assume role already normalized to F/D
    fig = px.histogram(
        peak_df, x="peak_delta", color="role",
        category_orders={"role": ["F", "D"]},
        color_discrete_map={"F": "#ff7f0e", "D": "#1f77b4"},
        barmode="overlay", nbins=40, opacity=0.6,
        template="plotly_white", title=title,
        labels={"peak_delta": "Peak Δ", "role": "Role"},
    )
    fig.update_layout(legend_traceorder="normal")
    return fig


In [24]:
# Self
SELF_COL_CANDIDATES = ["spicy_self", "spicy_score", "spicy_weighted", "spicy_w_dz", "spicy_self_delta"]
self_col = next((c for c in SELF_COL_CANDIDATES if c in df_z.columns), None)
if not self_col:
    raise KeyError(f"Need a self metric in df_z. Looked for: {SELF_COL_CANDIDATES}")

pd_self = compute_peak_delta(df_z, value_col=self_col, group_cols=("player","peak_year"), role_col="role")
bar_topk_peak_delta(pd_self, k=5, title=f"Top 5 Peak Δ — Player vs SELF [{self_col}]").show()
hist_peak_delta(pd_self, "Peak Δ — Player vs SELF (distribution by role)").show()

# Cohort (position-aware spicy_score)
if "spicy_score" not in df_z_cohort.columns:
    raise KeyError("df_z_cohort must contain 'spicy_score' (position-aware cohort composite).")

pd_cohort = compute_peak_delta(df_z_cohort, value_col="spicy_score",
                               group_cols=("player","peak_year"), role_col="role")
bar_topk_peak_delta(pd_cohort, k=5, title="Top 5 Peak Δ — Player vs COHORT (position × rel_age)").show()
hist_peak_delta(pd_cohort, "Peak Δ — Player vs COHORT (distribution by role)").show()

